# Notebook 1 — Extraction NDVI Sentinel-2
## Zone : Mandoul, Tchad | Période : 2019-2023
### Objectif : extraire une série temporelle NDVI et l'exporter en CSV

In [1]:
!pip install pystac_client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 6.9 MB/s eta 0:00:00


In [2]:
!pip install planetary_computer

In [3]:
!pip install stackstac

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.3/64.3 kB 2.1 MB/s eta 0:00:00


# Importation des libraries

In [4]:
import pystac_client
import planetary_computer as pc
import stackstac
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Connexion au catalogue Satellite

In [6]:
catalog=pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=pc.sign_inplace)
print(catalog)

<Client id=microsoft-pc>


In [16]:
bbox = [17.0, 7.5, 18.5, 9.0]
periode = "2023-06-01/2023-10-31"

# Recherche des images Sentinel-2

In [27]:
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=periode,
    query={"eo:cloud_cover": {"lt": 20}},
    max_items=60
)
items= list(search.items())
print(f"Nombre d'images trouvées : {len(items)}")

Nombre d'images trouvées : 60


## Explore une image

In [28]:
item = items[0]
print(item.datetime)
print(item.properties["eo:cloud_cover"])

2023-10-29 09:01:01.024000+00:00
7.132869


In [29]:
for asset_key in item.assets:
    print(asset_key)

AOT
B01
B02
B03
B04
B05
B06
B07
B08
B09
B11
B12
B8A
SCL
WVP
visual
preview
safe-manifest
granule-metadata
inspire-metadata
product-metadata
datastrip-metadata
tilejson
rendered_preview


## Charger les images en Stack

In [30]:
stack = stackstac.stack(
    items,
    assets=["B04", "B08"],
    bounds = bbox,
    epsg=4326,
    resolution=0.001,
)
print(stack)

<xarray.DataArray 'stackstac-c02aea32f3fb4ebec1235d582d82242a' (time: 60,
                                                                band: 2,
                                                                y: 1500, x: 1500)> Size: 2GB
dask.array<fetch_raster_window, shape=(60, 2, 1500, 1500), dtype=float64, chunksize=(1, 1, 1024, 1024), chunktype=numpy.ndarray>
Coordinates: (12/44)
  * time                                     (time) datetime64[ns] 480B 2023-...
  * band                                     (band) <U3 24B 'B04' 'B08'
  * y                                        (y) float64 12kB 9.0 ... 7.501
  * x                                        (x) float64 12kB 17.0 17.0 ... 18.5
    id                                       (time) <U54 13kB 'S2B_MSIL2A_202...
    s2:high_proba_clouds_percentage          (time) float64 480B 1.285 ... 0....
    ...                                       ...
    proj:shape                               object 8B {10980}
    gsd                  

## Calculer le NDVI

In [31]:
red = stack.sel(band="B04")
nir = stack.sel(band="B08")
ndvi = (nir - red) / (nir + red + 1e-10)
print(ndvi)

<xarray.DataArray 'stackstac-c02aea32f3fb4ebec1235d582d82242a' (time: 60,
                                                                y: 1500, x: 1500)> Size: 1GB
dask.array<truediv, shape=(60, 1500, 1500), dtype=float64, chunksize=(1, 1024, 1024), chunktype=numpy.ndarray>
Coordinates: (12/39)
  * time                                     (time) datetime64[ns] 480B 2023-...
  * y                                        (y) float64 12kB 9.0 ... 7.501
  * x                                        (x) float64 12kB 17.0 17.0 ... 18.5
    id                                       (time) <U54 13kB 'S2B_MSIL2A_202...
    s2:high_proba_clouds_percentage          (time) float64 480B 1.285 ... 0....
    s2:mean_solar_azimuth                    (time) float64 480B 91.91 ... 138.1
    ...                                       ...
    s2:not_vegetated_percentage              (time) float64 480B 4.668 ... 1.316
    s2:product_uri                           (time) <U65 16kB 'S2B_MSIL2A_202...
    s2:p

## Extraire la série temporelle

In [33]:
ndvi_mean = ndvi.mean(dim=["x", "y"])
ndvi_mean = ndvi_mean.compute()
print(ndvi_mean)

KeyboardInterrupt: 

## Conversion en DataFrame propre

In [23]:
df = pd.DataFrame({
    "date": ndvi_mean.coords["time"].values,
    "ndvi_mean": ndvi_mean.values,
})
df.sort_values("date")
df.reset_index(drop= True)
print(df)

                      date  ndvi_mean
0  2023-10-17 09:08:59.024   0.281987
1  2023-10-17 09:08:59.024   0.282063
2  2023-10-17 09:08:59.024   0.352550
3  2023-10-17 09:08:59.024   0.352292
4  2023-10-22 09:10:21.024   0.372249
5  2023-10-22 09:10:21.024   0.380347
6  2023-10-22 09:10:21.024   0.380616
7  2023-10-22 09:10:21.024   0.362262
8  2023-10-22 09:10:21.024   0.362306
9  2023-10-22 09:10:21.024   0.340371
10 2023-10-22 09:10:21.024   0.417338
11 2023-10-22 09:10:21.024   0.416382
12 2023-10-22 09:10:21.024   0.416509
13 2023-10-29 09:01:01.024   0.390456
14 2023-10-29 09:01:01.024   0.342635
15 2023-10-29 09:01:01.024   0.346837
16 2023-10-29 09:01:01.024   0.347042
17 2023-10-29 09:01:01.024   0.359086
18 2023-10-29 09:01:01.024   0.359226
19 2023-10-29 09:01:01.024   0.408375


In [25]:
df['date'] = pd.to_datetime(df['date'])
df_grouped = df.groupby(df['date'].dt.date)['ndvi_mean'].mean().reset_index()
print(df_grouped)

         date  ndvi_mean
0  2023-10-17   0.317223
1  2023-10-22   0.383153
2  2023-10-29   0.364808


## Ancienne approche — trop lente avec une connexion limitée
## Remplacée par la boucle image par image ci-dessous

In [34]:
resultats = []

for item in items:
    try:
        mini_stack = stackstac.stack(
            [item],
            assets=["B04", "B08"],
            bounds=bbox,
            epsg=4326,
            resolution=0.01  # résolution réduite pour aller plus vite
        )
        red = mini_stack.sel(band="B04")
        nir = mini_stack.sel(band="B08")
        ndvi = (nir - red) / (nir + red + 1e-10)
        valeur = float(ndvi.mean().compute())
        resultats.append({
            "date": item.datetime.date(),
            "ndvi_mean": valeur
        })
        print(f"✅ {item.datetime.date()} → NDVI = {valeur:.3f}")
    except Exception as e:
        print(f"⚠️ {item.datetime.date()} → Erreur ignorée : {e}")

✅ 2023-10-29 → NDVI = 0.393


✅ 2023-10-29 → NDVI = 0.349


✅ 2023-10-29 → NDVI = 0.348


✅ 2023-10-29 → NDVI = 0.348


✅ 2023-10-29 → NDVI = 0.359


...

✅ 2023-10-12 → NDVI = 0.408


✅ 2023-10-12 → NDVI = 0.408


✅ 2023-10-12 → NDVI = 0.386


✅ 2023-10-12 → NDVI = 0.386


⚠️ 2023-10-12 → Erreur ignorée : Error reading Window(col_off=0, row_off=0, width=150, height=150) from 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/33/P/YK/2023/10/12/S2A_MSIL2A_20231012T090911_N0510_R050_T33PYK_20241108T033907.SAFE/GRANULE/L2A_T33PYK_A043376_20231012T092556/IMG_DATA/R10m/T33PYK_20231012T090911_B08_10m.tif?st=2026-03-20T03%3A49%3A02Z&se=2026-03-21T04%3A34%3A02Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-03-20T08%3A16%3A04Z&ske=2026-03-27T08%3A16%3A04Z&sks=b&skv=2025-07-05&sig=ZbBR%2Bsf7Yqr4khTj6x5tPAc7BY0lMfveb1ChBG5atFc%3D': RasterioIOError('Read failed. See previous exception for details.')
⚠️ 2023-10-12 → Erreur ignorée : Error opening 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/33/P/YK/2023/10/12/S2A_MSIL2A_20231012T090911_N0509_R050_T33PYK_20231012T170654.SAFE/GRANULE/L2A_T33PYK_A043376_20231012T092556/IMG_DATA/R10m/T33PYK_20231012T090911_B04_10m.tif?st=2026-

In [35]:
df_final = pd.DataFrame(resultats)
df_final['date'] = pd.to_datetime(df_final['date'])
df_final = df_final.groupby('date')['ndvi_mean'].mean().reset_index()
df_final = df_final.sort_values('date').reset_index(drop=True)
print(df_final)
print(f"\n{len(df_final)} dates uniques collectées")

# Exporter en CSV
df_final.to_csv('ndvi_mandoul_2023.csv', index=False)
print("✅ Fichier sauvegardé !")

        date  ndvi_mean
0 2023-10-12   0.397099
1 2023-10-14   0.420686
2 2023-10-17   0.318097
3 2023-10-22   0.383657
4 2023-10-29   0.366224

5 dates uniques collectées
✅ Fichier sauvegardé !
